# Final research table
Storage/LNG use a two-calendar-day revised-history proxy. ERA5-derived features and weather-adjusted storage-flow anomalies use six days. Forecasts use inferred issue date plus one day; run timestamps remain unverified. `surprise7` and `surprise7_z` retain compatibility names but mean weather-adjusted flow anomalies, not market-consensus surprises. Tradable labels enter the following settlement and exclude inferred rolls; source contract mapping is unverified.

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

ROOT = Path(os.environ.get("TTF_PROJECT_ROOT", Path.cwd().parent))
RAW = ROOT / "1. Raw Data"
OUT = ROOT / "3. Final_DB"
PROC = ROOT / "2. Data Processing"
funda = pd.read_csv(RAW / "funda.csv", parse_dates=["gasDayStart"])
wx = pd.read_csv(RAW / "weather_hist.csv", parse_dates=["date"])
funda = funda.sort_values("gasDayStart").reset_index(drop=True)
wx = wx.dropna(subset=["EU_HDD", "EU_CDD"]).sort_values("date").reset_index(drop=True)

In [2]:
feb29 = funda["gasDayStart"].dt.month.eq(2) & funda["gasDayStart"].dt.day.eq(29)
key = funda["gasDayStart"].dt.strftime("%m-%d").mask(feb29, "02-28")
year = funda["gasDayStart"].dt.year
hist = funda.loc[~feb29, ["gasDayStart", "full"]].copy()
hist["key"] = hist["gasDayStart"].dt.strftime("%m-%d")
hist["year"] = hist["gasDayStart"].dt.year
lookup = hist.set_index(["key", "year"])["full"]
prior = pd.DataFrame({lag: pd.MultiIndex.from_arrays([key, year - lag]).map(lookup) for lag in range(1, 6)})
funda["storage_dev"] = funda["full"] - prior.mean(axis=1, skipna=False)
funda["net_inj"] = -funda["netWithdrawal"] / 1000
stock_gap = funda["gasInStorage"].diff() - funda["net_inj"]
assert stock_gap.abs().median() < 0.1

In [3]:
feb29 = wx["date"].dt.month.eq(2) & wx["date"].dt.day.eq(29)
key = wx["date"].dt.strftime("%m-%d").mask(feb29, "02-28")
year = wx["date"].dt.year
hist = wx.loc[~feb29, ["date", "EU_HDD"]].copy()
hist["key"] = hist["date"].dt.strftime("%m-%d")
hist["year"] = hist["date"].dt.year
lookup = hist.set_index(["key", "year"])["EU_HDD"]
prior = pd.DataFrame({lag: pd.MultiIndex.from_arrays([key, year - lag]).map(lookup) for lag in range(1, 11)})
wx["hdd_anom"] = wx["EU_HDD"] - prior.mean(axis=1, skipna=False)
funda["storage_dev_lag"] = funda["storage_dev"].shift(1)
gas = funda[["gasDayStart", "net_inj", "storage_dev_lag"]].merge(wx[["date", "hdd_anom"]], left_on="gasDayStart", right_on="date", how="left")
gas = gas.drop(columns="date").sort_values("gasDayStart").reset_index(drop=True)

In [4]:
# Expanding fit on earlier complete gas days; at least one seasonal cycle.
month = gas["gasDayStart"].dt.month.to_numpy()
D = np.column_stack([np.ones(len(gas)), gas["hdd_anom"].to_numpy(), gas["storage_dev_lag"].to_numpy(), *[(month == m).astype(float) for m in range(2, 13)]])
y = gas["net_inj"].to_numpy()
complete = np.flatnonzero(np.isfinite(D).all(axis=1) & np.isfinite(y))
resid = np.full(len(gas), np.nan)
for j, t in enumerate(complete):
    if j < 365:
        continue
    rows = complete[:j]
    X = D[rows]
    beta, _, rank, _ = np.linalg.lstsq(X, y[rows], rcond=None)
    if rank == X.shape[1]:
        resid[t] = y[t] - D[t] @ beta
gas["weather_adjusted_flow"] = resid
gas["surprise7"] = gas["weather_adjusted_flow"].rolling(7, min_periods=7).sum()
past = gas["surprise7"].shift(1)
gas["surprise7_z"] = (gas["surprise7"] - past.rolling(365, min_periods=365).mean()) / past.rolling(365, min_periods=365).std().replace(0, np.nan)
gas["flow_source_gas_day"] = gas["gasDayStart"]
gas["flow_available_date"] = gas["flow_source_gas_day"] + pd.Timedelta(days=6)
gas["gasDayStart"] = gas["flow_available_date"]
gas = gas[["gasDayStart", "flow_source_gas_day", "flow_available_date", "weather_adjusted_flow", "surprise7", "surprise7_z"]]

In [5]:
px = pd.read_csv(PROC / "prices_processed.csv", parse_dates=["date"])
funda_p = pd.read_csv(PROC / "funda_processed.csv", parse_dates=["gasDayStart"])
lng = pd.read_csv(PROC / "lng_processed.csv", parse_dates=["gasDayStart"])
wx_p = pd.read_csv(PROC / "weather_hist_processed.csv", parse_dates=["date"])
fc = pd.read_csv(PROC / "weather_forecast_processed.csv", parse_dates=["date"])
px = px[px["date"].between("2016-01-04", "2026-10-07")].sort_values("date")
db = pd.merge_asof(px, funda_p.sort_values("gasDayStart"), left_on="date", right_on="gasDayStart", direction="backward", tolerance=pd.Timedelta("3D")).drop(columns="gasDayStart")
db = pd.merge_asof(db, lng.sort_values("gasDayStart"), left_on="date", right_on="gasDayStart", direction="backward", tolerance=pd.Timedelta("3D")).drop(columns="gasDayStart")
db = pd.merge_asof(db, gas.sort_values("gasDayStart"), left_on="date", right_on="gasDayStart", direction="backward", tolerance=pd.Timedelta("3D")).drop(columns="gasDayStart")
db = pd.merge_asof(db, wx_p.rename(columns={"date": "wx_date"}).sort_values("wx_date"), left_on="date", right_on="wx_date", direction="backward", tolerance=pd.Timedelta("3D")).drop(columns="wx_date")
db = db.merge(fc, on="date", how="left").sort_values("date").reset_index(drop=True)

In [6]:
# Labels must finish inside the selected research/backtest price calendar.
last_date = db["date"].max()
for h in [1, 5, 10]:
    inside = pd.to_datetime(db[f"exit_date{h}"]).le(last_date)
    db[f"trade_valid{h}"] = db[f"trade_valid{h}"] & inside
    db.loc[~inside, [f"ds12_trade{h}", f"ds23_trade{h}", f"ds13_trade{h}"]] = np.nan
legacy_inside = db["date"].shift(-5).notna()
db.loc[~legacy_inside, ["ds12_fwd5", "ds23_fwd5", "ds13_fwd5"]] = np.nan

assert db["date"].is_monotonic_increasing and not db["date"].duplicated().any()
assert db["storage_dev"].notna().all()
assert np.allclose(db["s13"], db["s12"] + db["s23"])
for prefix in ["storage", "lng", "flow", "wx", "fc"]:
    available = pd.to_datetime(db[f"{prefix}_available_date"])
    assert (available.isna() | available.le(db["date"])).all()
for h in [1, 5, 10]:
    cols = [f"ds12_trade{h}", f"ds23_trade{h}", f"ds13_trade{h}"]
    valid = db[cols].notna().all(axis=1)
    assert np.allclose(db.loc[valid, cols[2]], db.loc[valid, cols[0]] + db.loc[valid, cols[1]])
    assert pd.to_datetime(db.loc[valid, f"entry_date{h}"]).gt(db.loc[valid, "date"]).all()
    assert pd.to_datetime(db.loc[valid, f"exit_date{h}"]).le(last_date).all()
db.to_csv(OUT / "final_db.csv", index=False)
print(db.shape, db["date"].min(), db["date"].max())
print(db[["vol_m1", "sd_ds12", "sd_ds23", "surprise7", "surprise7_z", "fc_hdd_anom7", "ds12_trade1", "ds12_trade5", "ds12_trade10"]].count())

(2778, 102) 2016-01-04 00:00:00 2026-10-07 00:00:00
vol_m1          2778
sd_ds12         2778
sd_ds23         2778
surprise7       2511
surprise7_z     2253
fc_hdd_anom7     679
ds12_trade1     2518
ds12_trade5     1998
ds12_trade10    1353
dtype: int64
